In [9]:
# 02 - CNN Baseline
# Full flow: grid search for best settings -> save to config -> train final model -> evaluate on val and test

import sys
sys.path.insert(0, '..')

import os
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models.cnn_model import SimpleCNN
from src.data.split_dataset import check_dataset
from src.utils.metrics import get_predictions, show_confusion_matrix, show_classification_report

In [10]:
# check device - use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
# make sure all three folders exist before going further
train_dir = "../data/processed/train"
val_dir = "../data/processed/val"
test_dir = "../data/processed/test"

for folder in [train_dir, val_dir, test_dir]:
    if not os.path.isdir(folder):
        print(folder, "not found, run 01_data_prep.ipynb first")

In [12]:
# image size is fixed, not part of the grid search
image_size = 128

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

# load all three datasets - ImageFolder uses folder names as class labels
train_data = datasets.ImageFolder(train_dir, transform=transform)
val_data = datasets.ImageFolder(val_dir, transform=transform)
test_data = datasets.ImageFolder(test_dir, transform=transform)

num_classes = len(train_data.classes)
class_names = train_data.classes

print("Classes:", train_data.class_to_idx)
print("Train size:", len(train_data), "Val size:", len(val_data), "Test size:", len(test_data))

Classes: {'healthy': 0, 'low_tread': 1, 'sidewall_damaged': 2, 'uneven_wear': 3, 'zero_tread': 4}
Train size: 520 Val size: 60 Test size: 67


In [13]:
# quick class balance check on train set - should be roughly equal after augmentation
counts = check_dataset(train_dir)

Dataset check:
healthy : 104 images 
low_tread : 104 images 
sidewall_damaged : 104 images 
uneven_wear : 104 images 
zero_tread : 104 images 
Total: 520 images


In [17]:
# grid search - try a few combinations, keep the best one based on val accuracy

learning_rates = [0.01, 0.001]
batch_sizes = [16, 32]

best_val_acc = 0
best_settings = None

for lr in learning_rates:
    for bs in batch_sizes:
        print("Trying lr:", lr, "batch_size:", bs)

        train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=bs, shuffle=False)

        # fresh model each trial - don't reuse weights across trials
        model = SimpleCNN(num_classes).to(device)
        loss_function = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # short run per combination - just enough to compare
        for epoch in range(5):
            model.train()
            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = loss_function(outputs, labels)
                loss.backward()
                optimizer.step()

        # check val accuracy after these 10 epochs
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                predicted = outputs.argmax(dim=1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print("  val acc:", round(val_acc, 3))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_settings = {"learning_rate": lr, "batch_size": bs}

print()
print("Best settings:", best_settings, "with val acc:", round(best_val_acc, 3))

Trying lr: 0.01 batch_size: 16
  val acc: 0.3
Trying lr: 0.01 batch_size: 32
  val acc: 0.133
Trying lr: 0.001 batch_size: 16
  val acc: 0.45
Trying lr: 0.001 batch_size: 32
  val acc: 0.433

Best settings: {'learning_rate': 0.001, 'batch_size': 16} with val acc: 0.45


In [18]:
# save the winning settings to the config file automatically
# this closes the loop - the config file always matches what actually won the search

config = {
    "image_size": image_size,
    "batch_size": best_settings["batch_size"],
    "epochs": 40,
    "learning_rate": best_settings["learning_rate"]
}

with open('../configs/cnn_config.yaml', 'w') as f:
    yaml.dump(config, f)

print("Saved to configs/cnn_config.yaml:", config)

Saved to configs/cnn_config.yaml: {'image_size': 128, 'batch_size': 16, 'epochs': 40, 'learning_rate': 0.001}


In [19]:
# load settings back from the config file
# from here on, the config file is the single source of truth - nothing is hardcoded

with open('../configs/cnn_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

batch_size = config['batch_size']
epochs = config['epochs']
learning_rate = config['learning_rate']

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

print("Training with:", config)

Training with: {'batch_size': 16, 'epochs': 40, 'image_size': 128, 'learning_rate': 0.001}


In [20]:
# build the final model - fresh weights, not reused from the grid search
model = SimpleCNN(num_classes).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# # early stopping based on val accuracy improvement, not exact repeats
# patience = 5
# epochs_without_improvement = 0
# best_val_acc = 0

# for epoch in range(epochs):
#     # ... training code stays exactly the same ...
#     # ... validation code stays exactly the same ...

#     print("Epoch", epoch + 1, "- train loss:", round(train_loss, 3),
#           "train acc:", round(train_acc, 3), "val acc:", round(val_acc, 3))

#     # check if this is the best val accuracy so far
#     if val_acc > best_val_acc:
#         best_val_acc = val_acc
#         epochs_without_improvement = 0
#         # save the best version of the model so far
#         torch.save(model.state_dict(), "../results/cnn/model.pt")
#         print("  new best val acc, model saved")
#     else:
#         epochs_without_improvement += 1

#     if epochs_without_improvement >= patience:
#         print("Training stopped early - no val improvement for", patience, "epochs")
#         break

In [21]:
# final training loop with early stopping
# stops if train accuracy and val accuracy both stay the same for 5 epochs in a row

patience = 5
same_count = 0
last_train_acc = None
last_val_acc = None

for epoch in range(epochs):

    # training
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print("Epoch", epoch + 1, "- train loss:", round(train_loss, 3),
          "train acc:", round(train_acc, 3), "val acc:", round(val_acc, 3))

    # early stopping check
    no_change = (train_acc == last_train_acc) and (val_acc == last_val_acc)

    if no_change:
        same_count += 1
    else:
        same_count = 0

    last_train_acc = train_acc
    last_val_acc = val_acc

    if same_count >= patience:
        print("Training stopped early - no change for", patience, "epochs")
        break

Epoch 1 - train loss: 53.185 train acc: 0.217 val acc: 0.367
Epoch 2 - train loss: 52.863 train acc: 0.258 val acc: 0.25
Epoch 3 - train loss: 50.733 train acc: 0.317 val acc: 0.367
Epoch 4 - train loss: 46.945 train acc: 0.358 val acc: 0.467
Epoch 5 - train loss: 42.832 train acc: 0.462 val acc: 0.483
Epoch 6 - train loss: 38.767 train acc: 0.502 val acc: 0.45
Epoch 7 - train loss: 37.563 train acc: 0.521 val acc: 0.417
Epoch 8 - train loss: 35.561 train acc: 0.55 val acc: 0.433
Epoch 9 - train loss: 34.897 train acc: 0.531 val acc: 0.5
Epoch 10 - train loss: 33.693 train acc: 0.592 val acc: 0.483
Epoch 11 - train loss: 30.292 train acc: 0.629 val acc: 0.433
Epoch 12 - train loss: 30.052 train acc: 0.612 val acc: 0.483
Epoch 13 - train loss: 28.687 train acc: 0.644 val acc: 0.45
Epoch 14 - train loss: 28.976 train acc: 0.673 val acc: 0.4
Epoch 15 - train loss: 27.167 train acc: 0.662 val acc: 0.433
Epoch 16 - train loss: 25.371 train acc: 0.719 val acc: 0.45
Epoch 17 - train loss: 23.

In [ ]:
# reload the best saved weights before evaluating
# (the model in memory is from the last epoch trained, not necessarily the best one)
model.load_state_dict(torch.load("../results/cnn/model.pt"))
print("Loaded best saved weights for evaluation")

In [22]:
# check performance on val set - this is fine to look at during development
true_labels, predicted_labels = get_predictions(model, val_loader, device)

print("VAL SET RESULTS")
show_confusion_matrix(true_labels, predicted_labels, class_names)
show_classification_report(true_labels, predicted_labels, class_names)

VAL SET RESULTS
Confusion matrix
(rows = actual, columns = predicted)
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']
[[15  0  2  2  3]
 [ 1  2  2  1  5]
 [ 2  1  5  0  0]
 [ 0  3  1  2  2]
 [ 2  2  0  2  5]]
                  precision    recall  f1-score   support

         healthy       0.75      0.68      0.71        22
       low_tread       0.25      0.18      0.21        11
sidewall_damaged       0.50      0.62      0.56         8
     uneven_wear       0.29      0.25      0.27         8
      zero_tread       0.33      0.45      0.38        11

        accuracy                           0.48        60
       macro avg       0.42      0.44      0.43        60
    weighted avg       0.49      0.48      0.48        60



In [23]:
# check performance on TEST set - this is the final, honest, one-time score
# only run this once you are fully done tuning - do not go back and tune more after seeing this

test_true_labels, test_predicted_labels = get_predictions(model, test_loader, device)

print("TEST SET RESULTS (final)")
show_confusion_matrix(test_true_labels, test_predicted_labels, class_names)
show_classification_report(test_true_labels, test_predicted_labels, class_names)

TEST SET RESULTS (final)
Confusion matrix
(rows = actual, columns = predicted)
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']
[[17  2  4  0  0]
 [ 2  5  0  2  3]
 [ 2  0  7  0  0]
 [ 0  2  0  6  2]
 [ 2  3  0  1  7]]
                  precision    recall  f1-score   support

         healthy       0.74      0.74      0.74        23
       low_tread       0.42      0.42      0.42        12
sidewall_damaged       0.64      0.78      0.70         9
     uneven_wear       0.67      0.60      0.63        10
      zero_tread       0.58      0.54      0.56        13

        accuracy                           0.63        67
       macro avg       0.61      0.61      0.61        67
    weighted avg       0.63      0.63      0.63        67



In [24]:
# save the test results to a text file - small file, goes in git, this is the evidence trail
from sklearn.metrics import classification_report

report_text = classification_report(test_true_labels, test_predicted_labels, target_names=class_names)

os.makedirs("../results/cnn", exist_ok=True)

with open("../results/cnn/metrics.txt", "w") as f:
    f.write("Settings used: " + str(config) + "\n\n")
    f.write("Test set results:\n")
    f.write(report_text)

print("Saved to results/cnn/metrics.txt")

Saved to results/cnn/metrics.txt


In [25]:
# save the trained model - this file is too large for git, goes to Drive instead (see RULEBOOK.txt)
torch.save(model.state_dict(), "../results/cnn/model.pt")
print("Model saved to results/cnn/model.pt")

Model saved to results/cnn/model.pt
